# Этап 0. Математический инструментарий

Конспект: [[PSI-00-Math-Toolkit]]

Здесь нет оптики. Цель — руками пощупать четыре вещи, на которых держится вся статья:

1. гармоническое колебание и его **фаза**;
2. тождество $\cos(\varphi+\delta)$, которое превращает нелинейную задачу в линейную;
3. `arctan2` — как достать угол из пары $(c, s)$;
4. метод наименьших квадратов (МНК) для переопределённой системы.

В конце — маленький «спойлер» к статье: что будет, если подставить в МНК неверные сдвиги.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (8, 3.5)
plt.rcParams["axes.grid"] = True
rng = np.random.default_rng(0)

## 1. Гармоническое колебание

$$x(t) = A\cos(\omega t + \varphi)$$

- $A$ — амплитуда (размах от нуля),
- $\omega = 2\pi f$ — круговая частота,
- $\varphi$ — начальная фаза: насколько косинусоида сдвинута вдоль оси.

Фаза измеряется в радианах или градусах; сдвиг на $2\pi$ (360°) ничего не меняет.

In [ ]:
t = np.linspace(0, 2, 500)
A, f = 1.0, 1.0

fig, ax = plt.subplots()
for phi_deg in (0, 45, 90, 180):
    phi = np.deg2rad(phi_deg)
    ax.plot(t, A * np.cos(2 * np.pi * f * t + phi), label=f"φ = {phi_deg}°")
ax.set_xlabel("t, с")
ax.set_ylabel("x(t)")
ax.set_title("Одна и та же косинусоида с разной начальной фазой")
ax.legend(loc="upper right")
plt.show()

### Вращающийся вектор

Косинус — это проекция на горизонтальную ось точки, бегущей по окружности:

$$A\cos(\omega t+\varphi) = \mathrm{Re}\left[A\,e^{i(\omega t+\varphi)}\right]$$

Фаза — это просто **угол** этого вектора. Отсюда вся дальнейшая логика: чтобы найти фазу, нужно узнать обе координаты вектора (и косинус, и синус), а не одну.

In [ ]:
phi0 = np.deg2rad(30)
frames = 120
theta = np.linspace(0, 4 * np.pi, frames)  # ωt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 2]})
circle = np.linspace(0, 2 * np.pi, 200)
ax1.plot(np.cos(circle), np.sin(circle), color="0.7")
ax1.set_aspect("equal")
ax1.set_xlim(-1.3, 1.3)
ax1.set_ylim(-1.3, 1.3)
ax1.set_title("Вектор $e^{i(\\omega t+\\varphi)}$")
vector, = ax1.plot([], [], "o-", lw=2)
proj, = ax1.plot([], [], "r--", lw=1)

ax2.plot(theta, np.cos(theta + phi0), color="0.8")
ax2.set_xlabel("ωt, рад")
ax2.set_title("Его проекция на ось x — косинус")
trace, = ax2.plot([], [], "r")
dot, = ax2.plot([], [], "ro")


def update(i):
    z = np.exp(1j * (theta[i] + phi0))
    vector.set_data([0, z.real], [0, z.imag])
    proj.set_data([z.real, z.real], [z.imag, 0])
    trace.set_data(theta[: i + 1], np.cos(theta[: i + 1] + phi0))
    dot.set_data([theta[i]], [z.real])
    return vector, proj, trace, dot


anim = FuncAnimation(fig, update, frames=frames, interval=40, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 2. Тождество, которое делает задачу линейной

$$\cos(\varphi+\delta) = \cos\varphi\cos\delta - \sin\varphi\sin\delta$$

Умножим на $b$ и обозначим $c = b\cos\varphi$, $s = b\sin\varphi$:

$$b\cos(\varphi+\delta) = c\cos\delta - s\sin\delta$$

Слева $b$ входит линейно (множитель), а $\varphi$ спрятана внутри косинуса — из-за неё задача нелинейная. Справа неизвестные $c$ и $s$ входят **линейно** — просто как коэффициенты при известных $\cos\delta$ и $\sin\delta$. Это ровно переход от формулы (3) к формуле (4) в статье.

In [ ]:
b = 0.8
phi = np.deg2rad(50)
delta = np.linspace(0, 2 * np.pi, 400)

c = b * np.cos(phi)
s = b * np.sin(phi);
left = b * np.cos(phi + delta)
right = c * np.cos(delta) - s * np.sin(delta)
print("max |левая − правая| =", np.max(np.abs(left - right)))

fig, ax = plt.subplots()
ax.plot(np.rad2deg(delta), c * np.cos(delta), "--", label="c·cos δ")
ax.plot(np.rad2deg(delta), -s * np.sin(delta), "--", label="−s·sin δ")
ax.plot(np.rad2deg(delta), left, "k", lw=2, label="b·cos(φ+δ) — их сумма")
ax.set_xlabel("δ, градусы")
ax.legend()
plt.show()

# Обратный переход: из (c, s) обратно в (b, φ)
print("b =", np.hypot(c, s), " φ =", np.rad2deg(np.arctan2(s, c)), "°")

## 3. `arctan` против `arctan2`

Из $c = b\cos\varphi$ и $s = b\sin\varphi$ следует $\operatorname{tg}\varphi = s/c$ — формула (7) статьи.
Но обычный арктангенс видит только отношение и не различает $(c, s)$ и $(-c, -s)$, поэтому возвращает угол лишь в пределах $(-90°, 90°)$.
`arctan2(s, c)` смотрит на знаки обоих аргументов и возвращает полный угол в $(-180°, 180°]$.

In [ ]:
phi_true = np.linspace(-np.pi, np.pi, 721)
c =  np.cos(phi_true)
s = np.sin(phi_true)

with np.errstate(divide="ignore", invalid="ignore"):
    phi_atan = np.arctan(s / c)
phi_atan2 = np.arctan2(s, c)

fig, ax = plt.subplots()
ax.plot(np.rad2deg(phi_true), np.rad2deg(phi_atan2), lw=3, label="arctan2(s, c)")
ax.plot(np.rad2deg(phi_true), np.rad2deg(phi_atan), ".", ms=2, label="arctan(s / c)")
ax.set_xlabel("истинный угол, градусы")
ax.set_ylabel("восстановленный угол, градусы")
ax.legend()
plt.show()

Даже `arctan2` не может вернуть угол больше 180°: фаза 370° и фаза 10° неразличимы. Отсюда «несшитая фаза» в статье — к ней вернёмся на этапе 6.

## 4. Метод наименьших квадратов

### 4.1. Разминка: прямая

Есть точки $(x_i, y_i)$, хотим $y = kx + m$. Каждая точка даёт уравнение, неизвестных два, точек много — система **переопределена** и точного решения не имеет. МНК ищет такие $k, m$, при которых сумма квадратов невязок минимальна.

В матричном виде $A\mathbf{p} \approx \mathbf{y}$, где строки $A$ — это $[x_i,\ 1]$. Решение даёт **нормальное уравнение**:

$$A^{T}A\,\mathbf{p} = A^{T}\mathbf{y}$$

In [ ]:
x = np.linspace(0, 10, 25)
y = 0.7 * x + 2.0 + rng.normal(0, 0.6, x.size)

A = np.column_stack([x, np.ones_like(x)])

p_normal = np.linalg.solve(A.T @ A, A.T @ y)        # нормальное уравнение «руками»
p_lstsq, *_ = np.linalg.lstsq(A, y, rcond=None)     # то же, но численно устойчивее

print("нормальное уравнение: k, m =", p_normal)
print("np.linalg.lstsq:      k, m =", p_lstsq)
print(A)

fig, ax = plt.subplots()
ax.plot(x, y, "o", label="измерения")
ax.plot(x, A @ p_lstsq, "r", label=f"МНК: y = {p_lstsq[0]:.2f}x + {p_lstsq[1]:.2f}")
ax.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3], dtype=float)
y = np.array([2, 3, 7], dtype=float)
k = np.sum(x*y) / np.sum(x*x)          # 29/14 ≈ 2.0714

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(11,4))

xs = np.linspace(0, 3.5, 100)
ax1.plot(xs, k*xs, 'b-', label=fr"$\hat y = {k:.4f}\,x$")
ax1.scatter(x,y, color='red', zorder=5, label="данные")
for xi, yi in zip(x,y):
  ax1.plot([xi,xi], [yi,k*xi], 'g--', lw=1)
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.set_title("Модель: прямая через ноль")
ax1.legend(); ax1.grid(True)

ks = np.linspace(1.5, 2.6, 200)
S = lambda k: np.sum((y - np.multiply.outer(k, x))**2, axis=-1)
ax2.plot(ks, S(ks), 'b-', label=r"$S(k)=\sum(y_i-kx_i)^2$")
ax2.axvline(k, color='red', ls='--', label=fr"$k^*={k:.4f}$")
ax2.scatter([k], [S(k)], color='red', zorder=5)
ax2.set_xlabel("k"); ax2.set_ylabel("S(k)")
ax2.set_title("Функционал ошибки: парабола")
ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.show()


A = x.reshape(-1,1)
print(A)
k_lstsq, *_ = np.linalg.lstsq(A, y, rcond=None)
print("k (lstsq) = ", k_lstsq[0])

### 4.2. То же самое, но для косинуса

Модель одной точки интерферограммы (формула (2) статьи):

$$I_n = a + b\cos(\varphi + \delta_n), \qquad n = 1 \dots N$$

Сдвиги $\delta_n$ известны, измерения $I_n$ известны, ищем $a, b, \varphi$. После замены из раздела 2:

$$I_n = a\cdot 1 + c\cdot\cos\delta_n + s\cdot(-\sin\delta_n)$$

Это та же линейная задача, что и с прямой, только столбцы матрицы другие: $[1,\ \cos\delta_n,\ -\sin\delta_n]$ — матрица (6) статьи.

In [ ]:
a_true, b_true, phi_true = 2.0, 0.8, np.deg2rad(50)
noise = 0.05

N = 7
delta = np.sort(rng.uniform(0, 2 * np.pi, N))  # сдвиги произвольные, но известные
I = a_true + b_true * np.cos(phi_true + delta) + rng.normal(0, noise, N)


def fit_cosine(delta, I):
    # МНК-подгонка I = a + b·cos(φ + δ). Возвращает a, b, φ.
    A = np.column_stack([np.ones_like(delta), np.cos(delta), -np.sin(delta)])
    print(A)
    (a, c, s), *_ = np.linalg.lstsq(A, I, rcond=None)
    return a, np.hypot(c, s), np.arctan2(s, c)


a, b, phi = fit_cosine(delta, I)
print(f"истина:  a = {a_true:.3f}  b = {b_true:.3f}  φ = {np.rad2deg(phi_true):.2f}°")
print(f"МНК:     a = {a:.3f}  b = {b:.3f}  φ = {np.rad2deg(phi):.2f}°")
print(f"модуляция γ = b/a = {b / a:.3f}")

d = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots()
ax.plot(np.rad2deg(d), a_true + b_true * np.cos(phi_true + d), "0.7", lw=4, label="истина")
ax.plot(np.rad2deg(d), a + b * np.cos(phi + d), "r", label="МНК")
ax.plot(np.rad2deg(delta), I, "ko", label="измерения")
ax.set_xlabel("δ, градусы")
ax.set_ylabel("I")
ax.legend()
plt.show()

### 4.3. Анимация: чем больше измерений, тем точнее

Три точки — минимум (три неизвестных). Каждая следующая усредняет шум.

In [ ]:
N_max = 30
delta_all = rng.uniform(0, 2 * np.pi, N_max)
I_all = a_true + b_true * np.cos(phi_true + delta_all) + rng.normal(0, 0.15, N_max)

fig, ax = plt.subplots()
ax.plot(np.rad2deg(d), a_true + b_true * np.cos(phi_true + d), "0.7", lw=4, label="истина")
points, = ax.plot([], [], "ko")
curve, = ax.plot([], [], "r", label="МНК")
title = ax.set_title("")
ax.set_xlim(0, 360)
ax.set_ylim(a_true - 1.5, a_true + 1.5)
ax.set_xlabel("δ, градусы")
ax.legend(loc="upper right")


def update(i):
    n = i + 3
    a, b, phi = fit_cosine(delta_all[:n], I_all[:n])
    points.set_data(np.rad2deg(delta_all[:n]), I_all[:n])
    curve.set_data(np.rad2deg(d), a + b * np.cos(phi + d))
    err = np.rad2deg(np.angle(np.exp(1j * (phi - phi_true))))
    title.set_text(f"N = {n},  ошибка фазы = {err:+.1f}°")
    return points, curve, title


anim = FuncAnimation(fig, update, frames=N_max - 2, interval=400, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

### 4.4. Насколько точнее? Численный эксперимент

Для равномерных сдвигов $\delta_n = 2\pi n / N$ теория даёт простую оценку ошибки фазы (в радианах):

$$\sigma_\varphi \approx \frac{\sigma_I}{b}\sqrt{\frac{2}{N}}$$

Проверим методом Монте-Карло: много раз генерируем шум и смотрим разброс найденной фазы.

In [ ]:
trials = 4000
Ns = np.arange(3, 31)
sigma_mc = []

for N in Ns:
    delta = 2 * np.pi * np.arange(N) / N
    A = np.column_stack([np.ones(N), np.cos(delta), -np.sin(delta)])
    I = a_true + b_true * np.cos(phi_true + delta) + rng.normal(0, noise, (trials, N))
    coef = np.linalg.lstsq(A, I.T, rcond=None)[0]  # все trials решаются одним вызовом
    phi = np.arctan2(coef[2], coef[1])
    sigma_mc.append(np.std(phi - phi_true))

fig, ax = plt.subplots()
ax.plot(Ns, np.rad2deg(sigma_mc), "o", label="Монте-Карло")
ax.plot(Ns, np.rad2deg(noise / b_true * np.sqrt(2 / Ns)), "r", label=r"$\sigma_I/b\cdot\sqrt{2/N}$")
ax.set_xlabel("число измерений N")
ax.set_ylabel("СКО ошибки фазы, градусы")
ax.legend()
plt.show()

## 5. Спойлер к статье: что если сдвиги известны неверно?

До сих пор мы подставляли в матрицу настоящие $\delta_n$. Возьмём сдвиги из статьи — реальные 0°, 72°, 198°, 266,4°, 36° — а в МНК подставим «паспортные» 0°, 90°, 180°, 270°, 360°. Шума нет вообще.

In [ ]:
delta_real = np.deg2rad([0, 72, 198, 266.4, 36])
delta_assumed = np.deg2rad([0, 90, 180, 270, 360])

phi_scan = np.linspace(-np.pi, np.pi, 721)
I = a_true + b_true * np.cos(phi_scan[:, None] + delta_real[None, :])  # строки — разные φ


def fit_many(delta, I):
    A = np.column_stack([np.ones_like(delta), np.cos(delta), -np.sin(delta)])
    coef = np.linalg.lstsq(A, I.T, rcond=None)[0]
    return np.arctan2(coef[2], coef[1])


def wrap(x):
    return np.angle(np.exp(1j * x))


err_wrong = wrap(fit_many(delta_assumed, I) - phi_scan)
err_right = wrap(fit_many(delta_real, I) - phi_scan)

fig, ax = plt.subplots()
ax.plot(np.rad2deg(phi_scan), np.rad2deg(err_wrong), label="паспортные сдвиги (шаг 90°)")
ax.plot(np.rad2deg(phi_scan), np.rad2deg(err_right), label="реальные сдвиги")
ax.set_xlabel("истинная фаза φ, градусы")
ax.set_ylabel("ошибка найденной фазы, градусы")
ax.legend()
plt.show()

print(f"размах ошибки при неверных сдвигах: {np.rad2deg(np.ptp(err_wrong)):.1f}°")

Ошибка не случайная, а **систематическая**: она зависит от самой фазы и повторяется дважды за период. На интерферограмме это даст рябь с удвоенной частотой полос — те самые «ступеньки» с рис. 5 статьи. Как найти реальные сдвиги по самим кадрам — и есть содержание статьи (этап 8).

## Упражнения

1. В разделе 4.2 поставьте `N = 3`, затем `N = 2`. Что происходит и почему?
2. Возьмите три сдвига 0°, 1°, 2°. Посчитайте `np.linalg.cond(A)` и ошибку фазы. Сравните с 0°, 120°, 240°. Вывод: сдвиги должны быть «разнесены» по окружности.
3. Выведите нормальное уравнение сами: продифференцируйте $\sum_n (I_n - a - c\cos\delta_n + s\sin\delta_n)^2$ по $a$, $c$, $s$ и приравняйте к нулю.
4. В разделе 5 подставьте свои «реальные» сдвиги (например, шаг 105° вместо 90°). Как размах ошибки зависит от величины ошибки шага?
5. Подставьте в разделе 4.2 столбец $+\sin\delta$ вместо $-\sin\delta$. Какой знак получит фаза и как поправить формулу для $\varphi$?